In [1]:
from Search import Search
from NormalizeTexts import NormalizeTexts
import pandas as pd
import urllib3 , warnings
from rapidfuzz import distance
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="elasticsearch._sync.client")

C:\Users\Asus\AppData\Roaming\Python\Python311\site-packages\langchain_huggingface\chat_models\__init__.py:1: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_huggingface.chat_models.huggingface import (


In [2]:

df = pd.read_csv(
    'Full_People_table_with_embeddings.csv',
    encoding='utf-8-sig',
    dtype={"شماره تلفن": str}
)


if 'full_information' not in df.columns:
    df['full_information'] = df['نام و نام خانوادگی'] + ' ' + df['نام واحد']


phone_series = pd.concat(
    [df['شماره تلفن'], df['شماره تلفن']],
    ignore_index=True
)


full_info_series = pd.concat(
    [df['full_information'], df['نام و نام خانوادگی']],
    ignore_index=True
)


df_new = pd.DataFrame({
    'full_information': full_info_series,
    'شماره تلفن': phone_series
})

print(df_new.head(10))
print(len(df_new))

                       full_information شماره تلفن
0       عالیه مورجانی امور مالی دانشگاه       2857
1              مهتاب وزیری دانشکده معدن       5117
2             سعید آباد آزمایشگاه مرکزی       5312
3        سعید آجلی دانشکده مهندسی نساجی       5039
4           سیدداود آذرشب جهاد دانشگاهی       2819
5                    حسن آذین فر بهداری       2949
6             محمدجواد آزادپرور تأسیسات       2230
7     مجتبی آقائی فروشانی دانشکده ریاضی       3648
8  کیوان آقابابایی سامانی دانشکده فیزیک       3763
9                فاطمه آقاجانی کتابخانه       2560
2924


In [3]:
data = df_new.to_dict()

In [4]:
data['full_information']

{0: 'عالیه مورجانی امور مالی دانشگاه',
 1: 'مهتاب وزیری دانشکده معدن',
 2: 'سعید آباد آزمایشگاه مرکزی',
 3: 'سعید آجلی دانشکده مهندسی نساجی',
 4: 'سیدداود آذرشب جهاد دانشگاهی',
 5: 'حسن آذین فر بهداری',
 6: 'محمدجواد آزادپرور تأسیسات',
 7: 'مجتبی آقائی فروشانی دانشکده ریاضی',
 8: 'کیوان آقابابایی سامانی دانشکده فیزیک',
 9: 'فاطمه آقاجانی کتابخانه',
 10: 'حسن آقاجانی دانشکده کشاورزی',
 11: 'عباس آقاجانی کوپائی پژوهشکده علوم و تکنولوژی زیردریا',
 12: 'مهران آقارخ گروه مهندسی تولید و ژنتیک -کشاورزی',
 13: 'مهران آقارخ دانشکده کشاورزی',
 14: 'فاطمه آقایی کتابخانه',
 15: 'محمد آهنگریان دانشکده فیزیک',
 16: 'مظاهر آیتی پور دانشکده کشاورزی',
 17: 'محمد اباذری دانشکده کشاورزی',
 18: 'سهیلا ابراهیمی کانون مشاوره خانواده',
 19: 'باقر ابراهیمی مرکز معارف',
 20: 'محمدصادق ابراهیمی دانشکده کشاورزی',
 21: 'اکبر ابراهیمی دانشکده برق و کامپیوتر',
 22: 'محمد ابراهیمی دانشکده برق و کامپیوتر',
 23: 'محمد صادق ابراهیمی گروه توسعه روستایی -کشاورزی',
 24: 'محمد رضا ابراهیمی خوابگاه ها',
 25: 'عیسی ابراهیمی 

In [5]:
header = 'full_information'

In [6]:
s = Search(data=data , url ="https://172.27.65.50:9200" 
           ,username="elastic"
            ,password="WhpSzbuJ1O=sJ3qjRTWL",
             verify_certs=False,
              index_name="people" )

c:\Users\Asus\.conda\envs\nlp\Lib\site-packages\elasticsearch\_sync\client\__init__.py:313: SecurityWarning: Connecting to 'https://172.27.65.50:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [7]:
queries = [
    "شمار تلفن، عالیهٔ مرجانی از امور مالی دانشگاه",
    "شماره تلفن آقای احمد نیکویی از دانشکده برق",
    "شماره تلفن آقای محمد جواد نصراصفهانی از دانشکده فیزیک",
    "شماره تلفن فاطمهٔ زهرانوربخش و نسفادرانی از امور خوابگاهها",
    "شماره تلفن خانم لهی ولایتی از دانشکدهی مواد",
    "شماره تلفن آقای سیب مهدی هاشمی جزی از فضای سبز",
    "شمارهی تلفن دکتر فخرالدین اشرفی زاده از دانشکدهی مواد دانشگاه صنعتی اصفهان",
    "شماره تلفن آقای جاسم ارجمندزاده از مرکز فنآوری اطلاعات دانشگاه صنعتی اصفهان",
    "شماره تلفن دکتر مهدی احمدیان نجفعبادی از دانشکده مواد",
    #"شماره تلفن آزمایشگاه آموزش عمومی و جی ای اساس گروه خاک شناسی کشاورزی",
    #"شماره تلفن آزه کارتوگرافی از دانشکدهی منابع طبیعی",
    "شمارهی دکتر هوشنگ اصلی حارونی از دانشکدهی معدن",
    "شمارهی دکتر هوشنگ استیحارونی از دانشدهی معدن",
    "شماره تلفن دکتر مهدی احمدیان نجفعبادی از دانشکده مواد",
    "شماره تلفن محمد صادق ابراهیمی از دانشکدهی کشاورزی",
    "شماره تلفن داوت آذرشب از جهاد دانشگاهی",
    #"شماره تلفن آز کنترل خطی از دانشکدهی برق و کامپیوتر",
    #"شمارهی تلفن هستهی تحقیقاتی فنآوری اطلاعات هوش مصنوعی از جهاد دانشگاهی، دانشگاه صنعتی اصفهان",
    "شمارهی تلفن دکتر بصیری از مرکز آیتی دانشگاه صنعتی اصفهان",
    "شمارهی تلفن دکتر بسیری از مرکز آیتی دانشگاه صنعتی اصفهان",
    "شمارهی تلفن عالیهی مورجانی از آمور مالی دانشگاه",
    "شماره تلفن دکتر احمدزاده از دانشکده برگ",
    "شماره تلفن آقای سعید آباد از آزمشگاه مرکزی",
    "شماره تلفن دکتر امیر اخوان بی تکثیر از دانشکده برگ و کامپیوتر",
    "شماره تلفن آقای علیرضا اسماعیلزادی اصفهانی از انتظارمد",
    "شماره تلفن خانم لیلا امینی فرد از مدیریت برنامه بودزه و تشکیلات",
    "شماره تلفن آقای محمد رسا باغبانها از دانشکتی کشاورزی",
    "شماره تلفن دکتر محسن بدرسمای از دانشکده مکانیک",
    "شماره تلفن دکتر پالهنگ از دانشکدی برگ و کامپیوتر",
    "شماره تلفن خانم مریم قاسیمی خورزوگی از معاونت آموزشی",
    "شماره تلفن آقای سیدمهدی گریشی از دانشکده فنی مهندسی گلپایگان",
    "دکتر محناز بیات از دانشکد منابع طبیع",
    "دکتر محمود بهبودی، دانشکده ریاض",
    "دکتر امیر اعخوان",
    "خانم دکتر مرزی افواضل",
    "آی دکتر محسن بعد، دانشکد مکانیک",
    "دکتر محمدرضا احمدزاده دانشکده مهندسی برق کامپیوتر",
    "دکتر نسرین اعتصامی",
    "آی دکتر مسعود بهار از دانشکده کشاورزی",
    "خانم دکتر صدیق برهانی از رانشکده مستاجی",
    "آگای پژمان یوسفیان از اداره کارگزینی",
    "آگای احسان یوسفیان از امور تغذیه",
    "آگای پژمان یوسفیان نظف آبادی از اداره کارگزینه",
    "دکتر فرزاد هاشمزاده از دانشکده کشاورزی",
    "آگایی محمد و حابی از سهاد دانشگاهی",
    "دکتر مصطفی نوالبخش از پژوهشکدی علوم و تکنولوژی زیر دریا",
    "آگای وحید نسل از تاسیسات",
    "دکتر سید احمد میرهای از دانشکتی کشاورزی",
    "خانم نسلین محتبی از مرکز آموزشهای الکترونیکی و آزاد",
    "دکتر میهن منصوری اصفهانی از دانشکده معدن",
    "دکتر سیدرضا متحری بیگدلی از پژوهشکده فنآوری اطلاعات و ارتباع داد",
    "دکتر رضا مزروعی سبدانی از دانشکدی ریاضی",
    "دکتر موسم مداح علیه از دانشکده برگ و کامپیوتر",
    "دکتر رضا مختاری از دانشکدی ریاضی",
    "آگای جمشید محمدی از دانشکده شیمی",
    "خانم مریمالسادات محرک پور از اداره خوابگاه",
    "آگای اسماعیل مانیان از حوزه معابنت دانشجویی",
    "خانم فاطمه گلناری از دانشکده مهندسی شیمی",
    "دکتر گزدان کیوانی از دانشکده منابع طبیعه",
    "واقع مصطفی کوشکی از دانشگدی اومرا",
    "آیا احمد زاده",
    "آی احمدوند",
    "آیدکتر احمدوند",
    "آقای امیر اخوان",
    "خانم دکتر مینا امیری",
    "خانمینا امیری",
    "خانم نسلین اعتصامی",
    "آگای بهزاد کیامهر از اداره تامین به پشتیبانه",
    "خانم فیروز کوچکی از اداره رفاع کارکنه",
    "آگای ابراهیم کمحالی از جهاد دانشگاهی",
    "دکتر ظهره کشکولی از مرکز زبان دانشگاه",
    "دکتر مهدی کشمیری از دانشکتی مکانیک",
    "دکتر سید زفرالله کلانتری از دانشکده فیزیک",
    "حالگای جواد کریمی از دانشکده معدن",
    "خانم مریم کریمی از انتظامت",
    "دکتر سفورا کریمی از دانشجدی مهندسی شیمی",
    "فتالله کریمزاده از دانشکده موات",
    "دکتر کاظم کرمی از دانشکده شیمه",
    "دکتر پرویز کاملی از دانشگدهی فیزیک",
    "علیرضا کاظمی از دانشکده کشاورزی",
    "حاکای رسول قاضمی از دانشکده صنایع و برنامهریزی سیستمها",
    "خانم زهرقنایی از رفاع دانشجویان",
    "خانم سانایز گناهت از دانشگده او را",
    "خانم زهرا گزوینی از دانشکتی فنی مهندسی گلپایگان",
    "دفتر غلامرضا، قربانی از دانشکده گروه علوم دامی کشاورزی",
    "دکتر ناصر گدیری مدرس از دانشکده برگ و کامپیوتر",
    "پرعلی قاسمی از جهاد دانشگاهی",
    "دکتر محمد قانع از دانشکده مهندسی نسازه",
    "حاگای محسن قاسمی از گروه ماشینهای کشاورزی کشاورزی",
    "خانم لاله گاسمی از دانشکدی نسازی",
    "آیدکتر احمدوند",
    "آی دکتر مهدی تاتاری؟ دانشکده ریاض",
    "دکتر محمدرضا احمدزاده دانشکده برق",
    "آقای رضا قاسمی از کتابخانه",
    "دکتر عباس قایی از دانشکدهی مکانیک",
    "آقای همیدرزا فولاد چنگ از جهاد دانشگاهی",
    "دکتر فرحات فضیله از دانشکده فیزیک",
    "آگای کیومرز فرحبخش از مدیریت فنآوری، نوآوری و تتاریسازی",
    "دکتر نادر فتیانپور از دانشکده معدن",
    "دکتر القام فاضل از گروه آبیاری کشاورزی",
    "مصرفی قیور دانشکده مکانیک",
    "خانم شقایق نسیری حوزه ریاست دانشگاه",
    "آقای محمد نصیری از گروه علوم و صنای قضایی، کشاورزی",
    "دکتر کمیل نسوری دانشکده مهندسی نساجی",
    "دکتر مهران نهوی دانشکده مواد",
    "دکتر صفایانی دانشکده بر",
    "خانم سمیه نجفی دانشکده کشاورزی",
    "خانم نرگس نادری کیا جهاد دانشگاهی",
    "آقای قیصر علی میرزاباغری جهاد دانشگاهی",
    "آقای عباس میران زادهی پژوهشکدهی علوم و تکنولوژی زیردریا"
]


In [8]:
labels = [
    "شماره تلفن عالیه مورجانی از امور مالی دانشگاه",
    "شماره تلفن آقای احمد نیکویی از دانشکده برق",
    "شماره تلفن آقای محمد جواد نصر اصفهانی از دانشکده فیزیک",
    "شماره تلفن فاطمه زهرا نوربخش ورنوسفادرانی از امور خوابگاه‌ها",
    "شماره تلفن خانم الهه ولایتی از دانشکده مواد",
    "شماره تلفن آقای سیدمهدی هاشمی جزی از فضای سبز",
    "شماره تلفن دکتر فخرالدین اشرفی زاده از دانشکده مواد دانشگاه صنعتی اصفهان ",
    " شماره تلفن آقای جاسم ارجمندزاده از مرکز فناوری اطلاعات دانشگاه صنعتی اصفهان",
    " شماره تلفن دکتر مهدی احمدیان نجف ابادی از دانشکده مواد",
   # "شماره تلفن آزمایشگاه آموزش عمومی و GIS کروه خاک شناسی و کشاورزی از",
    #"شماره تلفن آز کارتوگرافی از دانشکده منابع طبیعی",
    "شماره  دکتر هوشنگ اسدی هارونی از دانشکده معدن",
     "شماره  دکتر هوشنگ اسدی هارونی از دانشکده معدن",
    " شماره تلفن دکتر مهدی احمدیان نجف آبادی از دانشکده مواد",
    "شماره تلفن محمد صادق ابراهیمی از دانشکده کشاورزی ",
    "شماره تلفن داود آذرشب از جهاد دانشگاهی ",
   # "شماره تلفن آز کنترل خطی از دانشکده برق و کامپیوتر",
   #  " شماره تلفن هسته تحقیقاتی فناوری اطلاعات-هوش مصنوعی از جهاد دانشگاهی دانشگاه صنعتی اصفهان",
    "شماره تلفن دکتر بصیری از مرکز آیتی دانشگاه صنعتی اصفهان",

    "شماره تلفن دکتر بصیری از مرکز آیتی دانشگاه صنعتی اصفهان",
    "شماره تلفن عالیه مورجانی از امور مالی دانشگاه",
    "شماره تلفن دکتر احمد زاده از دانشکده برق",
    "شماره تلفن آقای سعید آباد از آزمایشگاه مرکزی",
    "شماره تلفن دکتر امیر اخوان بی تقصیر از دانشکده برق و کامپیوتر",
    "شماره تلفن آقای  علیرضا اسماعیل زاده اصفهانی از انتظامات",
    "شماره تلفن خانم لیلا امینی فرد از مدیریت برنامه بودجه و تشکیلات",
    "شماره تلفن آقای محمدرضا باغبان‌ها از دانشکده کشاورزی",
    "شماره تلفن  دکتر محسن بدرسمای از دانشکده مکانیک",
    "شماره تلفن دکتر پالهنگ از دانشکده برق و کامپیوتر",
    "شماره تلفن خانم مریم قاسمی خورزوقی از معاونت آموزشی",
    "شماره تلفن آقای سید مهدی قریشی از دانشکده فنی مهندسی گلپایگان",
    "دکتر مهناز بیات از دانشکده منابع طبیعی",
    "دکتر محمود بهبودی دانشکده ریاضی ",
    "دکتر امیر اخوان",
    "خانم دکتر مرضیه افاضل",
    "اقای دکتر محسن بدر دانشکده مکانیک",
    "دکتر محمدرضا احمدزاده دانشکده مهندسی برق و کامپیوتر",
    "دکتر نسرین اعتصامی",
    "اقای دکتر مسعود بهار از دانشکده کشاورزی",
    "خانم دکتر صدیقه برهانی از دانشکده نساجی",
    "آقای پژمان یوسفیان از اداره کار گزینی",
    "آقای احسان یوسفیان از امور تغذیه",
    "آقای پژمان یوسفیان نجف آبادی از اداره کار گزینی",
    "دکتر فرزاد هاشم زاده از دانشکده کشاورزی",
    "آقای محمد وهابی از جهاد دانشگاهی",
    "دکتر مصطفی نور بخش از پژوهشکده علوم و تکنولوژی زیر دریا",
    "آقای وحید نصر از تاسیسات",
    "دکتر سید احمد میره‌ای از دانشکده کشاورزی",
    "خانم نسرین مهدوی از مرکز آموزش‌های الکترونیکی و آزاد",
    "دکتر میهن منصوری اصفهانی از دانشکده معدن",
    "دکتر سیدرضا مطهری بیدگلی از پژوهشکده فناوری اطلاعات و ارتباطات",
    "دکتر رضا مزروعی سبدانی از دانشکده ریاضی ",
    "دکتر محسن مداح علی از دانشکده برق و کامپیوتر",
    "دکتر رضا مختاری از دانشکده ریاضی",
    "آقای جمشید محمدی از دانشکده شیمی",
    "خانم مریم السادات محرک پور از اداره خوابگاه ها",
    "آقای اسماعیل مانیان از حوزه معاونت دانشجویی",
    "خانم فاطمه گلناری از دانشکده مهندسی شیمی ",
    "دکتر یزدان کیوانی از دانشکده منابع طبیعی",
    "آقای مصطفی کوشکی از دانشکده عمران",
    "آقای احمدزاده",
    "آقای احمدوند",
    "آقای دکتر احمدوند",
    "آقای امیر اخوان",
    "خانم دکتر مینا امیری",
    "خانم مینا امیری",
    "خانم نسرین اعتصامی",
    "آقای بهزاد کیانمهر از اداره تامین و پشتیبانی",
    "خانم فیروزه کوچکی از اداره رفاه کارکنان",
    "آقای ابراهیم کمالی از جهاد دانشگاهی",
    "دکتر زهره کشکولی از مرکز زبان دانشکاه",
    "دکتر مهدی کشمیری از دانشکده مکانیک",
    "دکتر سید ظفر الله کلانتری از دانشکده فیزیک",
    "آقای جواد کریمی از دانشکده معدن",
    "خانم مریم کریمی از انتظامات",
    "دکتر صفورا کریمی از دانشکده مهندسی شیمی",
    "فتح الله کریم زاده از دانشکده مواد",
    "دکتر کاظم کرمی از دانشکده شیمی",
    "دکتر پرویز کاملی از دانشکده فیزیک",
    "علیرضا کاظمی از دانشکده کشاورزی",
    "آقای رسول کاظمی از دانشکده صنایع و برنامه ریزی سیستم ها",
    "خانم زهرا قناعی از رفاه دانشجویان",
    "خانم ساناز قناعت از دانشکده عمران",
    "خانم زهرا قزوینی از دانشکده فنی مهندسی گلپایگان",
    "دکتر غلامرضا قربانی از دانشکده گروه علوم دامی کشاورزی",
    "دکتر ناصر قدیری مدرس از دانشکده برق و کامپیوتر",
    "دکتر علی قاسمی از جهاد دانشگاهی",
    "دکتر محمد قانع از دانشکده مهندسی نساجی ",
    "آقای محسن قاسمی از گروه ماشین های کشاورزی کشاورزی ",
    "خانم لاله قاسمی از دانشکده نساجی",
    "آقای دکتر احمدوند",
    "آقای دکتر مهدی تاتاری دانشکده ریاضی ",
    "دکتر محمدرضا احمد زاده دانشکده برق",
    "آقای رضا قاسمی از کتاب خانه",
    "دکتر عباس قایی از دانشکده مکانیک",
    "آقای حمید رضا فولاد چنگ از جهاد دانشگاهی",
    "دکتر فرهاد  فضیله از دانشکده فیزیک",
    "آقای کیومرث فرحبخش از مدیریت فناوری نوآوری و تجاری سازی",
    "دکتر نادر فتحیان پور از دانشکده معدن",
    "دکتر الهام فاضل از گروه آبیاری کشاورزی",
    "مصطفی غیور دانشکده مکانیک",
    "خانم شقایق نصیری حوزه ریاست دانشگاه",
    "آقای محمد نصیری از گروه علوم و صنایع غذایی کشاورزی",
    "دکتر کمیل نصوری دانشکده مهندسی نساجی",
    "دکتر مهران نحوی دانشکده مواد",
    "دکتر صفایانی دانشکده برق",
    "خانم سمیه نجفی دانشکده کشاورزی",
    "خانم نرگس نادری کیا جهاد دانشگاهی",
    "آقای قصیر علی میرزا باقری جهاد دانشگاهی",
    "آقای عباس میران زاده پژوهشکده علوم و تکنولوژی زیر دریا",


]

In [9]:
normal = NormalizeTexts()

In [10]:
results = []

for idx, q in enumerate(queries):
    temp = []
    print(q)
    res = s.search(q, header)
    if res is not None : 
        print(res[header])

        true_lablel = normal.normalize_names(labels[idx])
    
        for item in res[header].values():
            item_norm = normal.normalize_names(item)   
            
            #print(f'q is {q} | item is {item_norm}')

            score = distance.Levenshtein.normalized_similarity(true_lablel, item_norm)
            
            


            temp.append(score)
        print('\n')
        if temp : 
            max_score = max(temp)
        else :
            max_score = 0
        results.append(max_score)


شمار تلفن، عالیهٔ مرجانی از امور مالی دانشگاه
{0: 'عالیه مورجانی امور مالی دانشگاه'}


شماره تلفن آقای احمد نیکویی از دانشکده برق
{1388: 'منیره نکویی دانشکده برق و کامپیوتر'}


شماره تلفن آقای محمد جواد نصراصفهانی از دانشکده فیزیک
{1371: 'محمد جواد نصراصفهانی دانشکده فیزیک'}


شماره تلفن فاطمهٔ زهرانوربخش و نسفادرانی از امور خوابگاهها
{1396: 'فاطمه زهرا نوربخش ورنوسفادرانی خوابگاه ها'}


شماره تلفن خانم لهی ولایتی از دانشکدهی مواد
{1420: 'الهه ولایتی دانشکده مواد'}


شماره تلفن آقای سیب مهدی هاشمی جزی از فضای سبز
{1436: 'سیدمهدی هاشمی جزی فضای سبز'}


شمارهی تلفن دکتر فخرالدین اشرفی زاده از دانشکدهی مواد دانشگاه صنعتی اصفهان
{96: 'سیدفخرالدین اشرفی زاده دانشکده مواد'}


شماره تلفن آقای جاسم ارجمندزاده از مرکز فنآوری اطلاعات دانشگاه صنعتی اصفهان
{62: 'جاسم ارجمند زاده پژوهشکده فناوری اطلاعات و ارتباطات'}


شماره تلفن دکتر مهدی احمدیان نجفعبادی از دانشکده مواد
{54: 'مهدی احمدیان نجف ابادی دانشکده مواد'}


شمارهی دکتر هوشنگ اصلی حارونی از دانشکدهی معدن
{81: 'هوشنگ اسدی هارونی دانشکده معدن

In [11]:
len(results)

106

In [12]:
sum(results)/len(results)

0.8919046229157078